# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:17<00:00,  5.75s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

"Title: StackSocial September Spotlight Deals: Up to 94% off\nDetails: StackSocial's September Spotlight Deals cover a range of software and digital subscriptions, from VPN services to cloud storage and AI writing tools. AdGuard VPN's 1-year plan is priced at $14.97, while a lifetime subscription to Drime Secure Cloud Storage with 8TB of space runs $279.97. The sale ends September 20. Shop Now at StackSocial\nFeatures: Lifetime and multi-year software subscriptions included Deals span VPN services, cloud storage, AI writing tools, and PDF editors Microsoft Visual Studio Professional 2026 included in the lineup AdGuard VPN available in 1-, 3-, and 5-year subscription options Drime Secure Cloud Storage available in standard and 8TB Advanced+ plans\nURL: https://www.dealnews.com/Stack-Social-September-Spotlight-Deals-Up-to-94-off/22181004.html?iref=rss-c39"

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Nintendo Alarmo Sound Clock for $77 + free shipping
Details: This Nintendo Sound Clock: Alarmo is $33 off the regular price of $109.99 at Best Buy. It offers 35 scenes inspired by five Nintendo franchises and a motion sensor that lets you dismiss the alarm with gestures instead of buttons. My Best Buy members get free shipping on all orders (it's free to join). Buy Now at Best 

In [8]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Nintendo Alarmo Sound Clock is a themed bedside alarm clock featuring 35 animated scenes drawn from Super Mario Odyssey, The Legend of Zelda: Breath of the Wild, Splatoon 3, Pikmin 4, and Ring Fit Adventure. It offers multiple wake modes including Steady and Gentle, an hourly chime, and ‘Sleepy Sounds’ for nighttime ambiance. The unit includes a motion sensor that lets you snooze or stop alarms with gestures, a Button Mode for traditional controls, and a record feature that tracks nighttime movement and time spent in bed.', price=77.0, url='https://www.dealnews.com/Nintendo-Alarmo-Sound-Clock-for-77-free-shipping/22181001.html?iref=rss-c142'), Deal(product_description='Mupoer Rechargeable 1.5V AA lithium battery 8-pack ships with an integrated charging and storage case and delivers a steady 1.5V output suitable for sensitive devices like game controllers and cameras. Each AA is rated 3,000 mWh capacity, supports about 2,500 charge cycles, 

In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Nintendo Alarmo Sound Clock is a themed bedside alarm clock featuring 35 animated scenes drawn from Super Mario Odyssey, The Legend of Zelda: Breath of the Wild, Splatoon 3, Pikmin 4, and Ring Fit Adventure. It offers multiple wake modes including Steady and Gentle, an hourly chime, and ‘Sleepy Sounds’ for nighttime ambiance. The unit includes a motion sensor that lets you snooze or stop alarms with gestures, a Button Mode for traditional controls, and a record feature that tracks nighttime movement and time spent in bed.
77.0
https://www.dealnews.com/Nintendo-Alarmo-Sound-Clock-for-77-free-shipping/22181001.html?iref=rss-c142

Mupoer Rechargeable 1.5V AA lithium battery 8-pack ships with an integrated charging and storage case and delivers a steady 1.5V output suitable for sensitive devices like game controllers and cameras. Each AA is rated 3,000 mWh capacity, supports about 2,500 charge cycles, and the kit includes smart LED charging indicators and a USB-C port for roughly three-hou

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

DealSelection(deals=[Deal(product_description='The TP-Link Festa FS328GP is a 28-port Gigabit smart managed PoE switch designed for small to medium networks and rack installation. It provides 24 PoE+ Gigabit ports with a 250-watt total power budget (up to 30W per port) plus 4 Gigabit SFP uplink slots for fiber or high-speed links. The unit includes metal casing with dual fans for cooling, static routing and layer‑2/3 management features (VLAN, ACL, etc.), and cloud-based management via the Festa app for remote monitoring and configuration.', price=118.99, url='https://www.dealnews.com/TP-Link-Festa-FS328-GP-28-Port-Gigabit-Smart-Managed-Po-E-Switch-for-119-free-shipping/22180922.html?iref=rss-c39'), Deal(product_description='The TCL S5 75S571G is a 75-inch 4K UHD LED smart TV that supports Dolby Vision, HDR10+, and HLG for expanded contrast and color. It runs Google TV with built-in Chromecast and voice remote, includes three HDMI inputs (one with eARC) and Dolby Atmos audio for enhanc

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [14]:
load_dotenv(override=True)

True

In [15]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [16]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [17]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [18]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [19]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [20]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
20:15:37 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
INFO:LiteLLM:
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
20:15:40 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
